<a href="https://colab.research.google.com/github/RashiBista/practice1/blob/main/bow_vs_tfidf.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Text Vectorization: Bag-of-Words vs TF-IDF
### Turning text into numbers two different ways

Machine learning models can't read raw text — words must first be converted into numeric vectors. This notebook compares the two most fundamental **vectorization** techniques:

| Technique | Idea | Question it answers |
|-----------|------|---------------------|
| **Bag-of-Words (BoW)** | Count how often each word appears | *How many times does this word occur?* |
| **TF-IDF** | Weight counts by how rare/distinctive a word is | *How important is this word to this document?* |

We'll build both on the same small corpus, look at the actual matrices, inspect the IDF weights, verify the math by hand, and finally see how the choice changes a real downstream task (document similarity).

## 0. Setup

In [ ]:
# !pip install scikit-learn pandas numpy
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option('display.width', 120)
print("Ready.")

## 1. The corpus

A *corpus* is just a collection of documents. Here each document is a short sentence about Nepal. Notice that some words appear in many sentences (**is**, **the**, **Nepal**) while others are unique to one (**Everest**, **Pokhara**, **food**). That contrast is exactly what TF-IDF is built to exploit.

In [ ]:
corpus = [
    "Kathmandu is the capital of Nepal",
    "Pokhara is a beautiful city in Nepal",
    "I love the food in Kathmandu",
    "Nepal is famous for the Himalayas and Mount Everest",
]
labels = ["Doc1", "Doc2", "Doc3", "Doc4"]

for lbl, doc in zip(labels, corpus):
    print(f"{lbl}: {doc}")

## 2. Bag-of-Words (BoW) encoding

BoW does two things:

1. **Builds a vocabulary** — every unique word across all documents becomes a column.
2. **Counts occurrences** — each document becomes a row of word counts.

It's called a *bag* because word **order is thrown away** — only the counts remain. `"dog bites man"` and `"man bites dog"` get identical BoW vectors.

We use scikit-learn's `CountVectorizer`. By default it lowercases text and splits on word boundaries.

In [ ]:
count_vectorizer = CountVectorizer()
bow_matrix = count_vectorizer.fit_transform(corpus)

vocab = count_vectorizer.get_feature_names_out()
print("Vocabulary size:", len(vocab))
print("Vocabulary:", list(vocab))
print("\nMatrix shape (documents x vocabulary):", bow_matrix.shape)

### The BoW matrix

Each row is a document, each column a vocabulary word, each value a raw count. Most entries are `0` — this is why these matrices are stored as **sparse** matrices in practice.

In [ ]:
bow_df = pd.DataFrame(bow_matrix.toarray(),
                      columns=vocab,
                      index=labels)
bow_df

### The problem with raw counts

Look at the columns for **is**, **the**, and **nepal** — they're packed with `1`s because those words appear in almost every document. In BoW these high-frequency, low-information words get just as much weight as a rare, meaningful word like **everest**.

For most tasks (search, classification, similarity) that's backwards: a word shared by *every* document tells you almost nothing about which document is which. **TF-IDF fixes exactly this.**

In [ ]:
# How many documents does each word appear in? (document frequency)
doc_freq = (bow_df > 0).sum().sort_values(ascending=False)
print("Document frequency (how many of the 4 docs contain each word):\n")
print(doc_freq)

## 3. TF-IDF encoding

**TF-IDF** = **Term Frequency** × **Inverse Document Frequency**. It keeps the counting idea of BoW but multiplies each count by a weight that *shrinks* common words and *boosts* rare ones.

**Term Frequency (TF)** — how often a term appears in a document (the BoW count, often normalized).

**Inverse Document Frequency (IDF)** — how rare a term is across the whole corpus. scikit-learn computes it (with smoothing) as:

$$\text{idf}(t) = \ln\!\left(\frac{1 + N}{1 + \text{df}(t)}\right) + 1$$

where $N$ = number of documents and $\text{df}(t)$ = number of documents containing term $t$.

- A word in **every** document → df is high → idf ≈ 1 (small weight).
- A word in **one** document → df is low → idf is large (big weight).

The final score is `tf * idf`, and scikit-learn then **L2-normalizes** each document vector so longer documents don't dominate.

In [ ]:
tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(corpus)

tfidf_df = pd.DataFrame(tfidf_matrix.toarray(),
                        columns=tfidf_vectorizer.get_feature_names_out(),
                        index=labels)
tfidf_df.round(3)

### Inspecting the IDF weights

This is the heart of TF-IDF. Words appearing in many documents get a **low** IDF; words appearing in only one get a **high** IDF. Sorting the IDF values makes the ranking obvious.

In [ ]:
idf_scores = pd.Series(tfidf_vectorizer.idf_,
                       index=tfidf_vectorizer.get_feature_names_out())
idf_scores = idf_scores.sort_values()

print("IDF weights (low = common/unimportant, high = rare/distinctive):\n")
print(idf_scores.round(3))

Notice **nepal**, **is**, and **the** sit at the bottom (they appear in 3 of 4 documents), while distinctive words like **everest**, **himalayas**, **pokhara**, and **food** sit at the top. TF-IDF will give those rare words the strongest signal.

## 4. Verify the math by hand

TF-IDF isn't a black box. Let's reproduce scikit-learn's IDF for the word **kathmandu** manually.

- Corpus size: $N = 4$
- **kathmandu** appears in Doc1 and Doc3 → $\text{df} = 2$

In [ ]:
import math

N = len(corpus)
df_kathmandu = 2   # appears in Doc1 and Doc3

idf_manual = math.log((1 + N) / (1 + df_kathmandu)) + 1

idf_sklearn = idf_scores["kathmandu"]

print(f"Manual  idf(kathmandu) = ln((1+{N})/(1+{df_kathmandu})) + 1 = {idf_manual:.4f}")
print(f"sklearn idf(kathmandu) = {idf_sklearn:.4f}")
print("Match:", np.isclose(idf_manual, idf_sklearn))

## 5. BoW vs TF-IDF, side by side

Comparing the two encodings for a single document makes the difference concrete. Below is **Doc1** (*"Kathmandu is the capital of Nepal"*) under each scheme. Only non-zero entries are shown.

In [ ]:
compare = pd.DataFrame({
    "BoW (count)": bow_df.loc["Doc1"],
    "TF-IDF (weight)": tfidf_df.loc["Doc1"].round(3),
})
compare = compare[compare["BoW (count)"] > 0]
print("Doc1: 'Kathmandu is the capital of Nepal'\n")
print(compare)

All six words have a BoW count of `1` — BoW treats them as equally important. Under TF-IDF, **capital** and **of** (unique to this document) get the highest weights, while **is**, **the**, and **nepal** are down-weighted. That's the whole point: *same words, smarter weighting.*

## 6. Why it matters: document similarity

Vectorization isn't the goal — it feeds a downstream task. A common one is measuring how **similar** two documents are using **cosine similarity** (1 = identical direction, 0 = nothing in common). Let's compute it with both encodings and compare.

In [ ]:
bow_sim = cosine_similarity(bow_matrix)
tfidf_sim = cosine_similarity(tfidf_matrix)

print("Cosine similarity with BoW:\n")
print(pd.DataFrame(bow_sim.round(3), index=labels, columns=labels))

print("\nCosine similarity with TF-IDF:\n")
print(pd.DataFrame(tfidf_sim.round(3), index=labels, columns=labels))